# Physics-Constrained Uncertainty

Apply physical constraints to UQ predictions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from deepuq.constraints import PositivityConstraint, ConservationConstraint, MonotonicityConstraint, apply_constraints
from deepuq.types import UQResult

## Positivity Constraint

In [ ]:
# Create a UQResult with some negative means (e.g., concentration predictions)
x = np.linspace(0, 5, 50)
mean = np.sin(x) - 0.3  # Some values are negative
std = np.ones_like(mean) * 0.2

result_before = UQResult(mean=mean, std=std)
print(f"Before: {(result_before.mean < 0).sum()} negative values")
print(f"Min value: {result_before.mean.min():.3f}")

# Apply positivity constraint
constraint = PositivityConstraint()
result_after = constraint.apply(result_before)

print(f"
After: {(result_after.mean < 0).sum()} negative values")
print(f"Min value: {result_after.mean.min():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, res, title in zip(axes, [result_before, result_after], ["Before", "After"]):
    ax.plot(x, res.mean, "b-")
    ax.fill_between(x, res.mean - 2*res.std, res.mean + 2*res.std, alpha=0.3)
    ax.axhline(0, color="r", linestyle="--", alpha=0.5)
    ax.set_title(f"{title} Positivity Constraint")
plt.tight_layout()
plt.show()

## Conservation Constraint

In [ ]:
# Create UQResult representing a density field that should integrate to 1.0
x = np.linspace(0, 1, 100)
dx = x[1] - x[0]
mean = np.exp(-((x - 0.5)**2) / 0.1)  # Gaussian-like density
mean = mean + np.random.randn(100) * 0.05  # Add noise
std = np.ones_like(mean) * 0.1

result_before = UQResult(mean=mean, std=std)
weights = np.ones_like(mean) * dx  # Integration weights

print(f"Integral before: {(result_before.mean * weights).sum():.4f}")

# Apply conservation constraint
constraint = ConservationConstraint(weights=weights, conserved_quantity=1.0)
result_after = constraint.apply(result_before)

print(f"Integral after: {(result_after.mean * weights).sum():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, res, title in zip(axes, [result_before, result_after], ["Before", "After"]):
    ax.plot(x, res.mean, "b-")
    ax.fill_between(x, res.mean - 2*res.std, res.mean + 2*res.std, alpha=0.3)
    ax.set_title(f"{title} Conservation Constraint")
    ax.set_xlabel("x")
plt.tight_layout()
plt.show()

## Monotonicity Constraint

In [ ]:
# Create UQResult with non-monotone mean (e.g., CDF that should be monotone)
x = np.linspace(0, 5, 50)
mean = 1 - np.exp(-x) + 0.15 * np.sin(3 * x)  # Non-monotone
std = np.ones_like(mean) * 0.1

result_before = UQResult(mean=mean, std=std)

# Apply monotonicity constraint
constraint = MonotonicityConstraint()
result_after = constraint.apply(result_before)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, res, title in zip(axes, [result_before, result_after], ["Before", "After"]):
    ax.plot(x, res.mean, "b-", linewidth=2)
    ax.fill_between(x, res.mean - 2*res.std, res.mean + 2*res.std, alpha=0.3)
    ax.set_title(f"{title} Monotonicity Constraint")
    ax.set_xlabel("x")
plt.tight_layout()
plt.show()

## Combining Constraints

In [ ]:
# Apply multiple constraints sequentially
x = np.linspace(0, 5, 50)
mean = np.sin(x) - 0.2  # Has negatives, non-monotone
std = np.ones_like(mean) * 0.15

result = UQResult(mean=mean, std=std)

constraints = [PositivityConstraint(), MonotonicityConstraint()]
result_final = apply_constraints(result, constraints)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, mean, "b-", linewidth=2)
axes[0].fill_between(x, mean - 2*std, mean + 2*std, alpha=0.3)
axes[0].axhline(0, color="r", linestyle="--", alpha=0.5)
axes[0].set_title("Original")

axes[1].plot(x, result_final.mean, "b-", linewidth=2)
axes[1].fill_between(x, result_final.mean - 2*result_final.std, result_final.mean + 2*result_final.std, alpha=0.3)
axes[1].axhline(0, color="r", linestyle="--", alpha=0.5)
axes[1].set_title("After Positivity + Monotonicity")

plt.tight_layout()
plt.show()